In [47]:
#Setup
import pandas as pd
import numpy as np
import re
from IPython.display import display

# FILE PATHS
INPUT_XLSX = r"C:/Users/Admin/Downloads/Malaysian E-Banking Service Quality & Customer Satisfaction Survey (Responses).xlsx"

df_raw = pd.read_excel(INPUT_XLSX)


In [48]:
# Step 1: Data Exploration
# Preview
print("\nPreview (first 5 rows)")
display(df_raw.head())

# Dtypes
print("\nInfo:")
df_raw.info()

# Missing values count
print("\nMissing values per column:")
display(df_raw.isnull().sum().sort_values(ascending=False).rename("missing").to_frame())

# Duplicate responses (common check: full-row duplicate)
print("\nDuplicate rows:", df_raw.duplicated().sum())


Preview (first 5 rows)


,Timestamp,I have read the information above and agree to participate.,Age Group,Nationality,Ethnicity,Gender,Employment Status,Monthly Income (RM),Number of Bank Accounts,Experience Using E-Banking,...,The layout and menus are well organized and easy to follow.,"Visual elements such as colors, icons, and fonts make the interface appealing.",I am satisfied with the overall quality of my bank’s e-banking services.,My experience using the e-banking system meets my expectations.,I would recommend my bank’s e-banking services to others.,Have you faced any issues when using your bank’s e-banking services?,When did the issue occur (within the last 3 months)?,Which e-banking platform did you experience the issue on?,Which bank did you encounter the problem with?,What issue(s) did you experience? (Select all that apply)
0,2025-11-26 19:54:01.767,"Yes, I agree",18–20 years,Malaysian,Chinese,Female,Student,"Below 2,000",One,1 – 3 years,...,3.0,2.0,1.0,2.0,2.0,Yes,1–4 weeks ago,Web / Online Banking,Al Rajhi Bank Malaysia,"[REL] Transaction failure, [REL] Delayed confi..."
1,2025-11-26 19:54:46.725,"Yes, I agree",31–40 years,Malaysian,Chinese,Female,Employed,"4,001 – 6,000",One,More than 6 years,...,4.0,5.0,5.0,4.0,5.0,Yes,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...
2,2025-11-26 19:58:50.045,"Yes, I agree",31–40 years,Malaysian,Malay,Female,Employed,"Above 8,000",More than two,1 – 3 years,...,5.0,4.0,4.0,5.0,4.0,Yes,Within the last 7 days,Web / Online Banking,AEON Bank,"[REL] Transaction failure, [REL] Delayed confi..."
3,2025-11-26 20:01:06.017,"Yes, I agree",31–40 years,Malaysian,Malay,Male,Employed,"6,001 – 8,000",More than two,More than 6 years,...,5.0,4.0,4.0,5.0,4.0,Yes,1–4 weeks ago,Web / Online Banking,Bank Rakyat,"[REL] Transaction failure, [REL] Delayed confi..."
4,2025-11-26 20:08:17.798,"Yes, I agree",41–50 years,Malaysian,Indian,Female,Employed,"2,001 – 4,000",One,1 – 3 years,...,4.0,5.0,5.0,4.0,5.0,Yes,Within the last 7 days,Both,Citibank Malaysia,"[REL] Transaction failure, [REL] Transfer cred..."



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 43 columns):
 #   Column                                                                           Non-Null Count  Dtype         
---  ------                                                                           --------------  -----         
 0   Timestamp                                                                        491 non-null    datetime64[ns]
 1   I have read the information above and agree to participate.                      491 non-null    object        
 2   Age Group                                                                        486 non-null    object        
 3   Nationality                                                                      486 non-null    object        
 4   Ethnicity                                                                        486 non-null    object        
 5   Gender                                                          

,missing
What issue(s) did you experience? (Select all that apply),153
Which bank did you encounter the problem with?,153
Which e-banking platform did you experience the issue on?,153
When did the issue occur (within the last 3 months)?,153
The e-banking platform works smoothly across all my devices.,5
My bank protects my transaction details from unauthorized access.,5
I trust my bank’s digital services to handle data securely.,5
The e-banking interface is easy to understand and use.,5
I can navigate the e-banking platform confidently without guidance.,5
Performing any banking task on the website or app requires little effort.,5



Duplicate rows: 0


In [49]:
#Step 2: Attribute Renaming
# Normalize headers
def normalize_header(s):
    s = str(s)
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s)
    s = s.strip()
    s = s.replace("’", "'").replace("“", '"').replace("”", '"')
    return s

df_raw2 = df_raw.copy()
df_raw2.columns = [normalize_header(c) for c in df_raw2.columns]

#Build rename mapping
rename_map = {
    "Timestamp": "response_time",
    "I have read the information above and agree to participate.": "consent_agreed",
    "Age Group": "age_group",
    "Nationality": "nationality",
    "Ethnicity": "ethnicity",
    "Gender": "gender",
    "Employment Status": "employment_status",
    "Monthly Income (RM)": "monthly_income_rm",
    "Number of Bank Accounts": "num_bank_accounts",
    "Experience Using E-Banking": "ebanking_experience",
    "Frequency of E-Banking Use": "usage_frequency",
    "Main Device Used for E-Banking": "main_device",
    "Region / Residence": "residence_region",
    "Which bank do you use most often for your e-banking services?": "main_bank",
    "Have you faced any issues when using your bank’s e-banking services?": "issue_experience",
    "When did the issue occur (within the last 3 months)?": "issue_timing",
    "Which e-banking platform did you experience the issue on?": "issue_platform",
    "Which bank did you encounter the problem with?": "issue_bank",
    "What issue(s) did you experience?   (Select all that apply)": "issue_types_raw",
}

#Rename Likert items → REL/RES/EFF/PS/EOU/SA/WBD/CS
likert_map = {
    # Reliability
    "My bank’s digital banking services are delivered accurately and without errors.": "REL1",
    "My bank consistently provides the digital services it promises.": "REL2",
    "Transactions are completed correctly the first time.": "REL3",

    # Responsiveness
    "My bank responds promptly when I need assistance with digital services.": "RES1",
    "Digital services are processed quickly when requested.": "RES2",
    "My bank clearly informs me when online services will be completed.": "RES3",

    # Efficiency
    "It is easy and quick to complete transactions using my bank’s online platform.": "EFF1",
    "I can easily find the information or service I need on the platform.": "EFF2",
    "The digital platform loads and operates smoothly without delay.": "EFF3",

    # Privacy & Security
    "I feel safe providing my personal or financial information online.": "PS1",
    "My bank protects my transaction details from unauthorized access.": "PS2",
    "I trust my bank’s digital services to handle data securely.": "PS3",

    # Ease of Use
    "The e-banking interface is easy to understand and use.": "EOU1",
    "I can navigate the e-banking platform confidently without guidance.": "EOU2",
    "Performing any banking task on the website or app requires little effort.": "EOU3",

    # System Availability
    "I can access the e-banking system whenever I need it.": "SA1",
    "The pages and functions load quickly and rarely crash.": "SA2",
    "The e-banking platform works smoothly across all my devices.": "SA3",

    # Website Design
    "The e-banking website has an attractive and professional design.": "WBD1",
    "The layout and menus are well organized and easy to follow.": "WBD2",
    "Visual elements such as colors, icons, and fonts make the interface appealing.": "WBD3",

    # Customer Satisfaction
    "I am satisfied with the overall quality of my bank’s e-banking services.  ": "CS1",
    "My experience using the e-banking system meets my expectations.": "CS2",
    "I would recommend my bank’s e-banking services to others.": "CS3",
}

# 3) Normalize mapping keys
rename_map_norm  = {normalize_header(k): v for k, v in rename_map.items()}
likert_map_norm  = {normalize_header(k): v for k, v in likert_map.items()}
full_map_norm    = {**rename_map_norm, **likert_map_norm}

# 4) Apply renaming
df = df_raw2.rename(columns=rename_map_norm).rename(columns=likert_map_norm).copy()

# 5) Rename table (no Found_in_File)
full_map_norm = {**rename_map_norm, **likert_map_norm}
rename_table = pd.DataFrame(
    [(old, new) for old, new in full_map_norm.items()],
    columns=["Original", "Renamed"]
)

display(rename_table)

,Original,Renamed
0,Timestamp,response_time
1,I have read the information above and agree to...,consent_agreed
2,Age Group,age_group
3,Nationality,nationality
4,Ethnicity,ethnicity
5,Gender,gender
6,Employment Status,employment_status
7,Monthly Income (RM),monthly_income_rm
8,Number of Bank Accounts,num_bank_accounts
9,Experience Using E-Banking,ebanking_experience


In [50]:
#Step 3: Inclusion Filtering

df_before = len(df)

# Count how many match each exclusion rule
consent_no = (df["consent_agreed"].astype(str).str.strip() == "No, I do not agree").sum()
below_18 = (df["age_group"].astype(str).str.strip() == "Below 18 years").sum()
less_month = (df["usage_frequency"].astype(str).str.strip() == "Less than once a month").sum()

# Apply the filters
consent_no = (df["consent_agreed"] == "No, I do not agree").sum()
below_18 = (df["age_group"] == "Below 18 years").sum()
less_month = (df["usage_frequency"] == "Less than once a month").sum()

df = df[
    (df["consent_agreed"] != "No, I do not agree") &
    (df["age_group"] != "Below 18 years") &
    (df["usage_frequency"] != "Less than once a month")
].copy()

df_after = len(df)

print("Excluded: No consent =", consent_no)
print("Excluded: Below 18 years =", below_18)
print("Excluded: Less than once a month =", less_month)
print("Rows before filter:", df_before)
print("Rows after filter:", df_after)
print("Total removed:", df_before - df_after)

Excluded: No consent = 5
Excluded: Below 18 years = 3
Excluded: Less than once a month = 57
Rows before filter: 491
Rows after filter: 426
Total removed: 65


In [51]:
# Step 4: Value Standardisation

# Ethnicity: keep Malay/Chinese/Indian else "Other"
allowed_ethnicity = {"Malay", "Chinese", "Indian"}

df["ethnicity"] = df["ethnicity"].astype(str).str.strip()
df["ethnicity"] = df["ethnicity"].where(df["ethnicity"].isin(allowed_ethnicity), "Other")

# Bank standardisation for main_bank and issue_bank
approved_banks = {
    "Maybank",
    "CIMB Bank",
    "Public Bank",
    "RHB Bank",
    "Hong Leong Bank",
    "Bank Islam Malaysia",
    "Bank Rakyat",
    "AmBank",
    "UOB Malaysia",
    "OCBC Bank",
    "Alliance Bank",
    "Affin Bank",
    "HSBC Bank Malaysia",
    "Standard Chartered Bank Malaysia",
    "Agrobank",
    "BSN",
    "MBSB Bank",
    "Bank of China Malaysia",
    "ICBC Malaysia",
    "Citibank Malaysia",
    "GXBank",
    "AEON Bank",
    "Boost Bank",
    "SeaBank Malaysia",
    "Al Rajhi Bank Malaysia"
}

def standardize_bank(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    return s if s in approved_banks else "Other"

df["main_bank"] = df["main_bank"].apply(standardize_bank)
df["issue_bank"] = df["issue_bank"].apply(standardize_bank)


# Ethnicity counts (fixed order)
eth_counts = (
    df["ethnicity"]
    .value_counts(dropna=False)
    .reindex(["Malay", "Chinese", "Indian", "Other"], fill_value=0)
    .reset_index()
)
eth_counts.columns = ["ethnicity", "count"]
display(eth_counts)

# Helper: force "Other" to appear (and optionally put it last)
def counts_with_other(series, label_col):
    vc = series.value_counts(dropna=False)

    # Ensure "Other" shows up even if 0
    if "Other" not in vc.index:
        vc.loc["Other"] = 0

    # Optional: move "Other" to the bottom
    vc = vc.drop("Other").sort_values(ascending=False)
    vc.loc["Other"] = series.eq("Other").sum()

    out = vc.reset_index()
    out.columns = [label_col, "count"]
    return out

# main_bank and issue_bank counts (include Other even if 0)
display(counts_with_other(df["main_bank"], "main_bank"))
display(counts_with_other(df["issue_bank"], "issue_bank"))

print("Ethnicity total:", df["ethnicity"].value_counts(dropna=False).sum())
print("Main bank total:", df["main_bank"].value_counts(dropna=False).sum())
print("Issue bank total:", df["issue_bank"].value_counts(dropna=False).sum())


,ethnicity,count
0,Malay,143
1,Chinese,191
2,Indian,92
3,Other,0


,main_bank,count
0,Agrobank,30
1,Standard Chartered Bank Malaysia,22
2,OCBC Bank,22
3,GXBank,21
4,RHB Bank,20
5,Affin Bank,20
6,Citibank Malaysia,20
7,HSBC Bank Malaysia,19
8,ICBC Malaysia,19
9,SeaBank Malaysia,18


,issue_bank,count
0,NaN,126
1,Bank of China Malaysia,20
2,HSBC Bank Malaysia,17
3,Affin Bank,16
4,Alliance Bank,15
5,Standard Chartered Bank Malaysia,15
6,CIMB Bank,14
7,SeaBank Malaysia,14
8,UOB Malaysia,14
9,Agrobank,13


Ethnicity total: 426
Main bank total: 426
Issue bank total: 426


In [52]:
# Step 5: Data Cleaning
# Structural Missing Handling (+ flag inconsistencies + blank out inconsistent follow-ups)

followup_fields = ["issue_timing", "issue_platform", "issue_bank", "issue_types_raw"]

def compute_followup_status(row):
    exp = row.get("issue_experience")
    exp = "" if pd.isna(exp) else str(exp).strip()

    followup_has_any = any(
        pd.notna(row.get(f)) and str(row.get(f)).strip() != ""
        for f in followup_fields
    )

    if exp == "No":
        return "Inconsistent" if followup_has_any else "Not Applicable"
    if exp == "Yes":
        return "Applicable" if followup_has_any else "Missing"
    return "Missing"  # unexpected label

# 1) Create followup_status
df["followup_status"] = df.apply(compute_followup_status, axis=1)

# 2) If "No" but follow-up has values -> flag is already "Inconsistent"
#    Now blank out the follow-up entries to align with Google Forms logic
incon_mask = df["followup_status"].eq("Inconsistent")
df.loc[incon_mask, followup_fields] = pd.NA

# become "Not Applicable" after cleaning
df["followup_status"] = df.apply(compute_followup_status, axis=1)

# QA summaries
display(df["issue_experience"].value_counts(dropna=False).to_frame("count"))
display(df["followup_status"].value_counts(dropna=False).to_frame("count"))

# Inspect inconsistent rows
display(df.loc[incon_mask, ["issue_experience"] + followup_fields].head(30))
print("Total inconsistent rows:", incon_mask.sum())


,count
issue_experience,
Yes,303
No,123


,count
followup_status,
Applicable,279
Not Applicable,123
Missing,24


,issue_experience,issue_timing,issue_platform,issue_bank,issue_types_raw
102,No,<NA>,<NA>,<NA>,<NA>
105,No,<NA>,<NA>,<NA>,<NA>
108,No,<NA>,<NA>,<NA>,<NA>
112,No,<NA>,<NA>,<NA>,<NA>
114,No,<NA>,<NA>,<NA>,<NA>
119,No,<NA>,<NA>,<NA>,<NA>
131,No,<NA>,<NA>,<NA>,<NA>
132,No,<NA>,<NA>,<NA>,<NA>
145,No,<NA>,<NA>,<NA>,<NA>
146,No,<NA>,<NA>,<NA>,<NA>


Total inconsistent rows: 21


In [53]:
# Handle missing values
objective1_cols = (
    [f"REL{i}" for i in range(1, 4)] +
    [f"RES{i}" for i in range(1, 4)] +
    [f"EFF{i}" for i in range(1, 4)] +
    [f"PS{i}"  for i in range(1, 4)] +
    [f"EOU{i}" for i in range(1, 4)] +
    [f"SA{i}"  for i in range(1, 4)] +
    [f"WBD{i}" for i in range(1, 4)] +
    [f"CS{i}"  for i in range(1, 4)]
)

# BEFORE
before_rows = len(df)
before_missing_rows = df[objective1_cols].isna().any(axis=1).sum()

print("Before deletion:")
print("Total rows:", before_rows)
print("Rows that have missing in Objective 1 columns:", before_missing_rows)

# DELETE (listwise deletion)
df_obj1 = df.dropna(subset=objective1_cols).copy()

# AFTER
after_rows = len(df_obj1)

print("\nAfter deletion:")
print("Total rows:", after_rows)
print("Rows removed:", before_rows - after_rows)
print("\nObjective 1 rows after listwise deletion:", len(df_obj1))


Before deletion:
Total rows: 426
Rows that have missing in Objective 1 columns: 0

After deletion:
Total rows: 426
Rows removed: 0

Objective 1 rows after listwise deletion: 426


In [54]:
import re
import numpy as np
import pandas as pd

# 1) Copy + make response_id
df_dash = df.reset_index(drop=True).copy()
df_dash["response_id"] = df_dash.index + 1

# 2) Keep only the columns you need for Tableau issues table
issues = df_dash[[
    "response_id", "issue_experience", "followup_status",
    "issue_timing", "issue_platform", "issue_bank", "issue_types_raw",
    "main_bank", "ethnicity", "age_group", "gender"
]].copy()

# 3) Keep only people who should have issue rows (Yes cases)
issues = issues[(issues["issue_experience"] == "Yes") & (issues["followup_status"] == "Applicable")]


# 4) Split checkbox string into rows (explode)
issues_long = (
    issues.assign(issue_option_raw=issues["issue_types_raw"].fillna("").str.split(","))
          .explode("issue_option_raw")
)

# 5) Clean spaces and drop blanks
issues_long["issue_option_raw"] = issues_long["issue_option_raw"].astype(str).str.strip()
issues_long = issues_long[issues_long["issue_option_raw"] != ""]

# 6) Extract tag like [REL] and remove it from label
#    - tag is letters inside [ ]
issues_long["issue_dimension_tag"] = issues_long["issue_option_raw"].str.extract(r"^\[([A-Z]+)\]")
issues_long["issue_label"] = issues_long["issue_option_raw"].str.replace(r"^\[[A-Z]+\]\s*", "", regex=True)

print("Issue long table rows:", len(issues_long))
display(issues_long.head(10))


Issue long table rows: 2956


,response_id,issue_experience,followup_status,issue_timing,issue_platform,issue_bank,issue_types_raw,main_bank,ethnicity,age_group,gender,issue_option_raw,issue_dimension_tag,issue_label
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[REL] Scheduled payment not processed as planned,REL,Scheduled payment not processed as planned
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[RES] Unhelpful chatbot / automated replies,RES,Unhelpful chatbot / automated replies
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[PS] Concern about data security,PS,Concern about data security
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[EOU] Hard-to-find functions,EOU,Hard-to-find functions
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[EOU] Difficulty logging in,EOU,Difficulty logging in
0,1,Yes,Applicable,1–4 weeks ago,Both,AEON Bank,[REL] Scheduled payment not processed as plann...,OCBC Bank,Chinese,31–40 years,Female,[WBD] Unattractive/confusing colour scheme,WBD,Unattractive/confusing colour scheme
1,2,Yes,Applicable,1–4 weeks ago,Web / Online Banking,Bank Rakyat,"[REL] Transaction failure, [REL] Delayed confi...",Citibank Malaysia,Malay,31–40 years,Male,[REL] Transaction failure,REL,Transaction failure
1,2,Yes,Applicable,1–4 weeks ago,Web / Online Banking,Bank Rakyat,"[REL] Transaction failure, [REL] Delayed confi...",Citibank Malaysia,Malay,31–40 years,Male,[REL] Delayed confirmation,REL,Delayed confirmation
1,2,Yes,Applicable,1–4 weeks ago,Web / Online Banking,Bank Rakyat,"[REL] Transaction failure, [REL] Delayed confi...",Citibank Malaysia,Malay,31–40 years,Male,[REL] Service outage,REL,Service outage
1,2,Yes,Applicable,1–4 weeks ago,Web / Online Banking,Bank Rakyat,"[REL] Transaction failure, [REL] Delayed confi...",Citibank Malaysia,Malay,31–40 years,Male,[REL] Transfer credited very late,REL,Transfer credited very late


In [55]:
import os

OUTPUT_DIR = r"C:/Users/Admin/Downloads/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

issues_long.to_excel(os.path.join(OUTPUT_DIR, "issues_long.xlsx"), index=False)
df_dash.to_excel(os.path.join(OUTPUT_DIR, "respondents_dash.xlsx"), index=False)
df_obj1.to_excel(os.path.join(OUTPUT_DIR, "objective1_cleaned.xlsx"), index=False)

print("Saved to:", OUTPUT_DIR)


Saved to: C:/Users/Admin/Downloads/
